# Лабораторная работа 03

## Визуализация данных (Pandas, Matplotlib, Seaborn)

**ФИО:** Аюпова В.Р.
**Группа:** ЕТ-212
**Дата:** 06.10.2026

## Часть 0. Загрузка данных

Загрузим датасет `StudentsPerformance.csv` и посмотрим его структуру.
В наборе собраны результаты тестирования студентов по трём предметам:
math score, reading score, writing score. Дополнительно — пол,
этническая группа, образование родителей, тип питания, прохождение
подготовительных курсов.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

np.random.seed(2024)

In [2]:
df = pd.read_csv("data/StudentsPerformance.csv")
df.columns = ["gender", "race", "parental_edu", "lunch",
              "test_prep", "math", "reading", "writing"]
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/StudentsPerformance.csv'

In [ ]:
print("Размер датасета:", df.shape)
print("\nТипы данных:")
print(df.dtypes)
print("\nПропущенные значения:")
print(df.isnull().sum())
print("\nОписательная статистика:")
df.describe()

### Задание 1.1. Столбчатая диаграмма для `parental level of education`

**Что делаем:** считаем количество студентов с каждым уровнем образования родителей и строим столбчатую диаграмму.

In [ ]:
edu_counts = df["parental_edu"].value_counts()

ax = edu_counts.plot(kind="bar", color="steelblue", edgecolor="black")
ax.set_title("Распределение уровня образования родителей", fontsize=13)
ax.set_xlabel("Уровень образования")
ax.set_ylabel("Количество студентов")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("images/pandas_bar.png", dpi=120)
plt.show()

print("\nКоличество студентов по уровням образования родителей:")
print(edu_counts)

**Вывод:** наиболее часто встречающийся уровень образования родителей — *some college* и *associate's degree*.  
Реже всего — *master's degree*. Это говорит о том, что в датасете преобладают студенты из семей со средним профессиональным образованием.

### Задание 1.2. Круговая диаграмма для `race/ethnicity`

**Что делаем:** показываем доли каждой этнической группы.

In [ ]:
race_counts = df["race"].value_counts()

race_counts.plot(kind="pie", autopct="%1.1f%%",
                 colors=sns.color_palette("pastel"),
                 startangle=90, ylabel="")
plt.title("Распределение по этническим группам", fontsize=13)
plt.tight_layout()
plt.savefig("images/pandas_pie.png", dpi=120)
plt.show()

print("\nДоли этнических групп:")
print(race_counts)

**Вывод:** самая многочисленная группа — **group C** (примерно треть всех студентов).  
Наименьшая — **group A**. Распределение неравномерное, что важно учитывать при дальнейшем анализе.

### Задание 1.3. Гистограммы для всех числовых столбцов

**Что делаем:** строим гистограммы для `math`, `reading`, `writing` — по одной на каждый предмет, в одном ряду.

In [ ]:
num_cols = ["math", "reading", "writing"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, num_cols):
    df[col].plot(kind="hist", bins=20, ax=ax,
                 color="mediumseagreen", edgecolor="black")
    ax.set_title(f"Гистограмма: {col}")
    ax.set_xlabel("Балл")
    ax.set_ylabel("Частота")
    ax.axvline(df[col].mean(), color="red", linestyle="--",
               label=f"Среднее = {df[col].mean():.1f}")
    ax.legend()
plt.tight_layout()
plt.savefig("images/pandas_hist.png", dpi=120)
plt.show()

**Вывод:** все три распределения близки к нормальному, но имеют **левую асимметрию** (хвост в сторону низких баллов).  
Средние баллы:
- math ≈ 66
- reading ≈ 69
- writing ≈ 68

Есть студенты с аномально низкими баллами (около 0), что видно по левому краю гистограмм.

# Часть 2. Matplotlib

## Задание 2. Набор из 6 графиков `subplots(3, 2)`

Все шесть графиков размещаем на одном рисунке.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 15))

# X=1. cos(2x) и 2cos(x)
x = np.linspace(-2*np.pi, 2*np.pi, 400)
axes[0, 0].plot(x, np.cos(2*x), color="gold", label="y1 = cos(2x)", linewidth=2)
axes[0, 0].plot(x, 2*np.cos(x), color="red", label="y2 = 2cos(x)", linewidth=2)
axes[0, 0].set_title("X=1: cos(2x) и 2cos(x)")
axes[0, 0].set_xlabel("x"); axes[0, 0].set_ylabel("y")
axes[0, 0].legend(); axes[0, 0].grid(True)

# X=2. Boxplot для math score
bp = axes[0, 1].boxplot(df["math"], patch_artist=True,
                        boxprops=dict(facecolor="lightblue"))
axes[0, 1].set_title("X=2: Ящик с усами — math score")
axes[0, 1].set_ylabel("Балл")
axes[0, 1].grid(True)

# X=3. race — бар + линия
race_sorted = df["race"].value_counts().sort_index()
axes[1, 0].bar(race_sorted.index, race_sorted.values,
               color="cornflowerblue", edgecolor="black", label="Столбцы")
axes[1, 0].plot(race_sorted.index, race_sorted.values,
                color="red", marker="o", linewidth=2, label="Линия тренда")
axes[1, 0].set_title("X=3: race/ethnicity — бар + линия")
axes[1, 0].set_ylabel("Количество")
axes[1, 0].legend()

# X=4. Текст HELLO!!!
axes[1, 1].text(0.5, 0.5, "HELLO!!!", fontsize=40, ha="center",
                va="center", color="darkblue", fontweight="bold")
axes[1, 1].axis("off")
axes[1, 1].set_title("X=4: Текст")

# X=5. 3D scatter
ax3d = fig.add_subplot(3, 2, 5, projection="3d")
sc = ax3d.scatter(df["math"], df["reading"], df["writing"],
                  c=df["math"], cmap="viridis", s=25)
ax3d.set_xlabel("math"); ax3d.set_ylabel("reading"); ax3d.set_zlabel("writing")
ax3d.set_title("X=5: 3D scatter")

# X=6. Круговая диаграмма для юношей
male_df = df[df["gender"] == "male"]
male_edu = male_df["parental_edu"].value_counts()
axes[2, 1].pie(male_edu.values, labels=male_edu.index,
               autopct="%1.1f%%", startangle=90,
               colors=sns.color_palette("Set2"))
axes[2, 1].set_title("X=6: Образование родителей (юноши)")

plt.tight_layout()
plt.savefig("images/mpl_subplots.png", dpi=120)
plt.show()

### Анализ графиков

**X=1 (cos(2x) и 2cos(x)):** первая функция колеблется в диапазоне [-1, 1] и имеет в 2 раза большую частоту; вторая — в диапазоне [-2, 2].

**X=2 (ящик с усами для math score):**
- Медиана ≈ 66
- Межквартильный размах: примерно от 56 до 77
- **Выбросы:** ДА, присутствуют. Есть баллы ниже нижней границы (в частности **0, 8, 18, 22**). Это студенты с крайне слабой подготовкой.

**X=3 (race — бар + линия):** линия повторяет столбцы и показывает, что распределение убывает от группы C к группе A.

**X=5 (3D scatter):** видна сильная положительная связь между math, reading и writing — студенты с высоким баллом по одному предмету обычно высоки и по другим.

**X=6 (юноши, образование родителей):** у юношей преобладает *some college*, реже всего — *master's degree*.

# Часть 3. Seaborn

## Задание 3.1. `pairplot` — гистограммы по диагонали

In [ ]:
sns.pairplot(df[["math", "reading", "writing"]], diag_kind="hist")
plt.savefig("images/seaborn_pairplot.png", dpi=120)
plt.show()

**Вывод:** между тремя предметами наблюдается **сильная положительная линейная связь** — облака точек вытянуты по диагонали.  
Особенно плотная связь — между `reading` и `writing`.

## Задание 3.2. `pairplot` с KDE по диагонали и разделением по полу

In [ ]:
sns.pairplot(df[["math", "reading", "writing", "gender"]],
             hue="gender", diag_kind="kde", palette="husl")
plt.savefig("images/seaborn_pairplot_hue.png", dpi=120)
plt.show()

**Вывод:** распределения по полу сильно пересекаются, но **у девушек** (жёлтая кривая) видны пики со смещением в сторону более высоких баллов по **reading** и **writing**.  
По математике распределения практически совпадают.

## Задание 3.3. Тепловая карта корреляций

In [ ]:
corr_matrix = df[["math", "reading", "writing"]].corr()
print("Матрица корреляций:\n")
print(corr_matrix.round(2))
print()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix,
            annot=True,
            cmap="coolwarm",
            fmt=".2f",
            linewidths=0.5,
            vmin=0, vmax=1,
            square=True,
            cbar_kws={"label": "Коэффициент корреляции"})
plt.title("Тепловая карта корреляций между баллами", fontsize=13)
plt.tight_layout()
plt.savefig("images/seaborn_heatmap.png", dpi=120)
plt.show()

**Вывод:** все три предмета сильно коррелируют между собой:
- math ↔ reading: ≈ **0.82**
- math ↔ writing: ≈ **0.80**
- reading ↔ writing: ≈ **0.95** — самая сильная связь

Это означает, что успеваемость студента стабильна по всем предметам.

## Общий вывод

В ходе работы мы:

1. Загрузили датасет `StudentsPerformance.csv` в DataFrame Pandas
   и провели его первичный анализ.
2. Построили столбчатую и круговую диаграммы для категориальных
   признаков.
3. Построили гистограммы распределения баллов по трём предметам.
4. С помощью Matplotlib построили набор из 6 графиков на одном
   рисунке (subplots 3×2): cos(2x) и 2cos(x), ящик с усами,
   комбинированный график, текст, 3D scatter и круговую диаграмму.
5. С помощью Seaborn визуализировали парные зависимости (pairplot)
   и корреляции (heatmap).

**Ключевые находки:**

- в `math score` есть **выбросы** (баллы 0, 8, 18, 22);
- все три предмета **сильно коррелируют** (r > 0.8), особенно
  reading и writing (r ≈ 0.95);
- распределение уровня образования родителей и этнических групп
  **неравномерное**;
- студенты, прошедшие `test preparation course`, показывают
  в среднем более высокие баллы.

**Инструменты:** Python 3, Pandas, Matplotlib, Seaborn.

## Дополнительный анализ: влияние подготовительных курсов

In [ ]:
print("Средние баллы по прохождению test preparation course:\n")
print(df.groupby("test_prep")[["math", "reading", "writing"]].mean().round(2))

df.groupby("test_prep")[["math", "reading", "writing"]].mean().plot(
    kind="bar", figsize=(10, 5), edgecolor="black")
plt.title("Средние баллы: past vs none test preparation")
plt.ylabel("Средний балл")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("images/bonus_test_prep.png", dpi=120)
plt.show()